# Claim Extraction-Test generation results

In [39]:
import pandas as pd
import matplotlib.pyplot as plt

results=pd.read_csv("summary_report.csv")
results.head()

results.columns


Index(['instance_id', 'claim_id', 'tests_model', 'claims_model', 'status',
       'discriminative', 'classification', 'attempts', 'bug', 'gold',
       'failure_reason', 'timestamp', 'slurm_job_id'],
      dtype='object')

In [40]:
# Lets check the classification results depending on the test model used
results.groupby(["tests_model","claims_model"])["classification"].value_counts().unstack()
# Add total instances tested for each model combination
results.groupby(["tests_model","claims_model"])["classification"].value_counts().unstack().fillna(0).assign(total=lambda x: x.sum(axis=1))



classification                                                                                         INVERTED  \
tests_model                                        claims_model                                                   
Qwen/Qwen2.5-72B-Instruct-AWQ                      Qwen/Qwen2.5-72B-Instruct-AWQ                            0.0   
                                                   Qwen/Qwen2.5-Coder-32B-Instruct-AWQ                      0.0   
                                                   hugging-quants/Meta-Llama-3.1-70B-Instruct-AWQ-...       0.0   
Qwen/Qwen2.5-Coder-32B-Instruct-AWQ                Qwen/Qwen2.5-72B-Instruct-AWQ                            0.0   
                                                   Qwen/Qwen2.5-Coder-32B-Instruct-AWQ                      1.0   
                                                   hugging-quants/Meta-Llama-3.1-70B-Instruct-AWQ-...       0.0   
deepseek-ai/DeepSeek-Coder-V2-Lite-Instruct        Qwen/Qwen2.5-Coder-32B-Instruct-AWQ                      0.0   
hugging-quants/Meta-Llama-3.1-70B-Instruct-AWQ-... Qwen/Qwen2.5-72B-Instruct-AWQ                            0.0   
                                                   Qwen/Qwen2.5-Coder-32B-Instruct-AWQ                      0.0   
                                                   hugging-quants/Meta-Llama-3.1-70B-Instruct-AWQ-...       0.0   

classification                                                                                         NON_DISCRIMINATIVE  \
tests_model                                        claims_model                                                             
Qwen/Qwen2.5-72B-Instruct-AWQ                      Qwen/Qwen2.5-72B-Instruct-AWQ                                      5.0   
                                                   Qwen/Qwen2.5-Coder-32B-Instruct-AWQ                                1.0   
                                                   hugging-quants/Meta-Llama-3.1-70B-Instruct-AWQ-...                 7.0   
Qwen/Qwen2.5-Coder-32B-Instruct-AWQ                Qwen/Qwen2.5-72B-Instruct-AWQ                                     11.0   
                                                   Qwen/Qwen2.5-Coder-32B-Instruct-AWQ                                1.0   
                                                   hugging-quants/Meta-Llama-3.1-70B-Instruct-AWQ-...                 4.0   
deepseek-ai/DeepSeek-Coder-V2-Lite-Instruct        Qwen/Qwen2.5-Coder-32B-Instruct-AWQ                                0.0   
hugging-quants/Meta-Llama-3.1-70B-Instruct-AWQ-... Qwen/Qwen2.5-72B-Instruct-AWQ                                      2.0   
                                                   Qwen/Qwen2.5-Coder-32B-Instruct-AWQ                                3.0   
                                                   hugging-quants/Meta-Llama-3.1-70B-Instruct-AWQ-...                10.0   

classification                                                                                         OVERCONSTRAINED  \
tests_model                                        claims_model                                                          
Qwen/Qwen2.5-72B-Instruct-AWQ                      Qwen/Qwen2.5-72B-Instruct-AWQ                                   4.0   
                                                   Qwen/Qwen2.5-Coder-32B-Instruct-AWQ                             7.0   
                                                   hugging-quants/Meta-Llama-3.1-70B-Instruct-AWQ-...              6.0   
Qwen/Qwen2.5-Coder-32B-Instruct-AWQ                Qwen/Qwen2.5-72B-Instruct-AWQ                                   4.0   
                                                   Qwen/Qwen2.5-Coder-32B-Instruct-AWQ                             6.0   
                                                   hugging-quants/Meta-Llama-3.1-70B-Instruct-AWQ-...             12.0   
deepseek-ai/DeepSeek-Coder-V2-Lite-Instruct        Qwen/Qwen2.5-Coder-32B-Instruct-AWQ                             2.0   
hugging-quants/Meta-

In [41]:
# Lets coun the number of claims that each model has extracted
results.groupby("claims_model")["claim_id"].count()

claims_model
Qwen/Qwen2.5-72B-Instruct-AWQ                          76
Qwen/Qwen2.5-Coder-32B-Instruct-AWQ                    55
hugging-quants/Meta-Llama-3.1-70B-Instruct-AWQ-INT4    87
Name: claim_id, dtype: int64

In [42]:
import os
import json

claims_folder = "/fs/nexus-scratch/ihbas/verifier_harness/claim_extraction/claims_out"
claim_models = os.listdir(claims_folder)

model_folders = [os.path.join(claims_folder, doc) for doc in claim_models if not doc.endswith(".json")]

claim_counts = {}

for path in model_folders:
  print(f"Processing model folder: {path}")
  claim_counts[path] = 0
  instances = os.listdir(path)
  for instance in instances:
      if instance == "summary.json": # Fixed typo from 'sumary.json'
        print("Summary found! Skipping...")
        continue
      
      file_path = os.path.join(path, instance)
      with open(file_path, 'r') as f:
          data = json.load(f)
          
          # Use instance_id as the key for your counts
          instance_id = data.get('instance_id', instance) 
          
          # The 'stats' key is the most reliable way to get the final count of each instance
          count = data.get("stats", {}).get('final_claims', 0)
          
          claim_counts[path] += count

# Display the results
print(claim_counts)

Processing model folder: /fs/nexus-scratch/ihbas/verifier_harness/claim_extraction/claims_out/Meta-Llama-3.1-70B-Instruct-AWQ-INT4
Summary found! Skipping...
Processing model folder: /fs/nexus-scratch/ihbas/verifier_harness/claim_extraction/claims_out/Qwen2.5-72B-Instruct-AWQ
Summary found! Skipping...
Processing model folder: /fs/nexus-scratch/ihbas/verifier_harness/claim_extraction/claims_out/Qwen2.5-Coder-32B-Instruct-AWQ
Summary found! Skipping...
{'/fs/nexus-scratch/ihbas/verifier_harness/claim_extraction/claims_out/Meta-Llama-3.1-70B-Instruct-AWQ-INT4': 32, '/fs/nexus-scratch/ihbas/verifier_harness/claim_extraction/claims_out/Qwen2.5-72B-Instruct-AWQ': 55, '/fs/nexus-scratch/ihbas/verifier_harness/claim_extraction/claims_out/Qwen2.5-Coder-32B-Instruct-AWQ': 22}


In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
from pathlib import Path
import warnings

# Suppress FutureWarnings
warnings.filterwarnings('ignore', category=FutureWarning)

# Set style for publication-quality plots
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 11

# Load data
df = pd.read_csv('summary_report.csv')
df= df[df['tests_model'] != 'deepseek-ai/DeepSeek-Coder-V2-Lite-Instruct']  # Filter out deepseek model
#Lets filter out pylint and django repos too
df = df[~df['instance_id'].str.contains('pylint|django|pytest', case=False, na=False)]

# Coerce discriminative to numeric once for downstream plots (fixed deprecation warning)
df['discriminative_num'] = pd.to_numeric(
    df['discriminative'].replace({True: 1, False: 0, 'True': 1, 'False': 0}).infer_objects(copy=False),
    errors='coerce'
)

# Create output directory
output_dir = Path('presentation_diagrams/analysis')
output_dir.mkdir(parents=True, exist_ok=True)

In [44]:
import math
import numpy as np
import matplotlib.pyplot as plt

def _short_model_name(m: str, max_len: int = 22) -> str:
    s = str(m).split("/")[-1]
    return (s[:max_len] + "…") if len(s) > max_len else s

def _model_grid(models, ncols=2):
    models = list(models)
    n = len(models)
    ncols = min(ncols, n) if n > 0 else ncols
    nrows = int(math.ceil(n / ncols)) if n > 0 else 1
    return nrows, ncols, models

def _make_model_subplots(models, title, ncols=2, figsize_per_ax=(6.2, 3.8)):
    nrows, ncols, models = _model_grid(models, ncols=ncols)
    fig, axes = plt.subplots(
        nrows, ncols,
        figsize=(figsize_per_ax[0] * ncols, figsize_per_ax[1] * nrows),
        squeeze=False
    )
    fig.suptitle(title, fontsize=16, fontweight="bold", y=0.98)
    return fig, axes, models, nrows, ncols

In [45]:
from pathlib import Path

def plot_classification_distribution_faceted(df, output_dir: Path, ncols=2):
    models = df["tests_model"].dropna().unique()
    fig, axes, models, nrows, ncols = _make_model_subplots(
        models,
        title="Classification Distribution (by Test Model)",
        ncols=ncols
    )

    # Consistent ordering across subplots
    all_classes = df["classification"].dropna().unique().tolist()
    # Optional: enforce a known order if you prefer
    preferred = ["VALID", "OVERCONSTRAINED", "NON_DISCRIMINATIVE", "UNRESOLVED", "INVERTED", "N/A"]
    classes = [c for c in preferred if c in all_classes] + [c for c in all_classes if c not in preferred]

    max_y = 0
    per_model_counts = {}

    for m in models:
        counts = df.loc[df["tests_model"] == m, "classification"].value_counts()
        per_model_counts[m] = counts
        max_y = max(max_y, counts.max() if len(counts) else 0)

    for i, m in enumerate(models):
        r, c = divmod(i, ncols)
        ax = axes[r][c]
        counts = per_model_counts[m]

        y = [counts.get(cls, 0) for cls in classes]
        x = np.arange(len(classes))

        ax.bar(x, y)
        ax.set_title(_short_model_name(m), fontsize=12, fontweight="bold")
        ax.set_xticks(x)
        ax.set_xticklabels(classes, rotation=20, ha="right", fontsize=9)
        ax.set_ylim(0, max_y * 1.15 if max_y > 0 else 1)
        ax.grid(axis="y", alpha=0.3)

        # annotate counts
        for xi, yi in zip(x, y):
            if yi > 0:
                ax.text(xi, yi + max_y * 0.03, str(int(yi)), ha="center", va="bottom", fontsize=9)

    # Hide empty axes
    for j in range(len(models), nrows * ncols):
        r, c = divmod(j, ncols)
        axes[r][c].axis("off")

    plt.tight_layout(rect=[0, 0, 1, 0.96])
    out = output_dir / "2_classification_distribution_by_model_grid.png"
    plt.savefig(out, dpi=300, bbox_inches="tight")
    plt.close()
    print(f"✓ Saved: {out.name}")

In [46]:
def plot_success_convergence_by_model(df, output_dir: Path, ncols=2):
    models = df["tests_model"].dropna().unique()
    fig, axes, models, nrows, ncols = _make_model_subplots(
        models,
        title="Success Convergence (Attempts to Success) by Test Model",
        ncols=ncols
    )

    for i, m in enumerate(models):
        r, c = divmod(i, ncols)
        ax = axes[r][c]
        mdf = df[(df["tests_model"] == m) & (df["status"] == "success")].copy()

        vals = mdf["attempts"].dropna().values
        ax.hist(vals, bins=range(1, 13), edgecolor="black", alpha=0.75)
        ax.set_title(_short_model_name(m), fontsize=12, fontweight="bold")
        ax.set_xlabel("Attempts", fontsize=10)
        ax.set_ylabel("Count", fontsize=10)
        ax.grid(axis="y", alpha=0.3)

        if len(vals):
            med = np.median(vals)
            ax.axvline(med, linestyle="--", linewidth=2)
            ax.text(med + 0.1, ax.get_ylim()[1] * 0.85, f"median={med:.1f}", fontsize=9)

    for j in range(len(models), nrows * ncols):
        r, c = divmod(j, ncols)
        axes[r][c].axis("off")

    plt.tight_layout(rect=[0, 0, 1, 0.96])
    out = output_dir / "3_success_convergence_by_model_grid.png"
    plt.savefig(out, dpi=300, bbox_inches="tight")
    plt.close()
    print(f"✓ Saved: {out.name}")

In [47]:
def plot_failure_modes_by_model(df, output_dir: Path, ncols=2):
    models = df["tests_model"].dropna().unique()
    fig, axes, models, nrows, ncols = _make_model_subplots(
        models,
        title="Failure Mode Distribution (by Test Model)",
        ncols=ncols
    )

    failure_mask = df["status"].isin(["failed", "non_discriminative"])
    failed_df = df[failure_mask].copy()

    # consistent class order
    preferred = ["OVERCONSTRAINED", "UNRESOLVED", "NON_DISCRIMINATIVE", "INVERTED"]
    all_classes = failed_df["classification"].dropna().unique().tolist()
    classes = [c for c in preferred if c in all_classes] + [c for c in all_classes if c not in preferred]

    max_y = 0
    per_model_counts = {}
    for m in models:
        counts = failed_df.loc[failed_df["tests_model"] == m, "classification"].value_counts()
        per_model_counts[m] = counts
        max_y = max(max_y, counts.max() if len(counts) else 0)

    for i, m in enumerate(models):
        r, c = divmod(i, ncols)
        ax = axes[r][c]
        counts = per_model_counts[m]

        y = [counts.get(cls, 0) for cls in classes]
        x = np.arange(len(classes))

        ax.bar(x, y)
        ax.set_title(_short_model_name(m), fontsize=12, fontweight="bold")
        ax.set_xticks(x)
        ax.set_xticklabels(classes, rotation=20, ha="right", fontsize=9)
        ax.set_ylim(0, max_y * 1.15 if max_y > 0 else 1)
        ax.grid(axis="y", alpha=0.3)

        for xi, yi in zip(x, y):
            if yi > 0:
                ax.text(xi, yi + max_y * 0.03, str(int(yi)), ha="center", va="bottom", fontsize=9)

    for j in range(len(models), nrows * ncols):
        r, c = divmod(j, ncols)
        axes[r][c].axis("off")

    plt.tight_layout(rect=[0, 0, 1, 0.96])
    out = output_dir / "6_failure_modes_by_model_grid.png"
    plt.savefig(out, dpi=300, bbox_inches="tight")
    plt.close()
    print(f"✓ Saved: {out.name}")

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from pathlib import Path
import warnings

# Suppress FutureWarnings
warnings.filterwarnings('ignore', category=FutureWarning)

# Set style for publication-quality plots
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 11

# Load data
df = pd.read_csv('summary_report.csv')
df= df[df['tests_model'] != 'deepseek-ai/DeepSeek-Coder-V2-Lite-Instruct']  # Filter out deepseek model
#Lets filter out pylint and django repos too
df = df[~df['instance_id'].str.contains('pylint|django|pytest', case=False, na=False)]

# Ensure discriminative_num is created for faceted plots
df['discriminative_num'] = pd.to_numeric(
    df['discriminative'].replace({True: 1, False: 0, 'True': 1, 'False': 0}).infer_objects(copy=False),
    errors='coerce'
)

plot_classification_distribution_faceted(df, output_dir, ncols=2)
plot_success_convergence_by_model(df, output_dir, ncols=2)
plot_failure_modes_by_model(df, output_dir, ncols=2)

In [49]:
import math
import numpy as np
import matplotlib.pyplot as plt

def plot_combo_success_faceted_by_test_model(
    df,
    output_path,
    min_n=3,
    ncols=2,
    max_claims=8,  # show top claims per test model by n
):
    d = df.copy()
    d["tests_short"]  = d["tests_model"].astype(str).str.split("/").str[-1].str[:22]
    d["claims_short"] = d["claims_model"].astype(str).str.split("/").str[-1].str[:22]

    # Aggregate per pair
    agg = (d.groupby(["tests_model", "claims_model", "tests_short", "claims_short"])
            .agg(success_rate=("discriminative", "mean"),
                 n=("discriminative", "size"))
            .reset_index())
    agg["success_rate"] *= 100

    tests = agg["tests_model"].unique().tolist()
    n = len(tests)
    if n == 0:
        print("⊘ No data")
        return

    ncols = min(ncols, n)
    nrows = int(math.ceil(n / ncols))

    fig, axes = plt.subplots(nrows, ncols, figsize=(7.2 * ncols, 3.8 * nrows), squeeze=False)
    fig.suptitle("Success Rate by (Claim Model → Test Model)", fontsize=16, fontweight="bold", y=0.99)

    for i, tm in enumerate(tests):
        r, c = divmod(i, ncols)
        ax = axes[r][c]

        sub = agg[agg["tests_model"] == tm].copy()
        sub = sub[sub["n"] >= min_n].copy()
        if sub.empty:
            ax.set_title(f"{sub['tests_short'].iloc[0] if len(sub) else tm.split('/')[-1]}\n(no pairs with n≥{min_n})",
                         fontsize=12, fontweight="bold")
            ax.axis("off")
            continue

        # keep the most supported claim models (readable)
        sub = sub.sort_values(["n", "success_rate"], ascending=[False, False]).head(max_claims)

        x = np.arange(len(sub))
        ax.bar(x, sub["success_rate"].values)
        ax.set_xticks(x)
        ax.set_xticklabels(sub["claims_short"].values, rotation=20, ha="right", fontsize=9)

        ax.set_ylim(0, 100)
        ax.set_ylabel("Success (%)", fontsize=10)
        ax.set_title(sub["tests_short"].iloc[0], fontsize=12, fontweight="bold")
        ax.grid(axis="y", alpha=0.3)

        for xi, (sr, nn) in enumerate(zip(sub["success_rate"].values, sub["n"].values)):
            ax.text(xi, sr + 2, f"n={int(nn)}", ha="center", fontsize=9)

    # hide unused axes
    for j in range(n, nrows * ncols):
        r, c = divmod(j, ncols)
        axes[r][c].axis("off")

    plt.tight_layout(rect=[0, 0, 1, 0.95])
    plt.savefig(output_path, dpi=300, bbox_inches="tight")
    plt.close()
    print(f"✓ Saved: {output_path}")

In [50]:
plot_combo_success_faceted_by_test_model(
    df,
    output_dir / "combo_success_faceted_by_test_model.png",
    min_n=3,
    ncols=2,
    max_claims=8
)

✓ Saved: presentation_diagrams/analysis/combo_success_faceted_by_test_model.png


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from pathlib import Path
import warnings

# Suppress FutureWarnings
warnings.filterwarnings('ignore', category=FutureWarning)

# Set style for publication-quality plots
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 11

# Load data
df = pd.read_csv('summary_report.csv')
df= df[df['tests_model'] != 'deepseek-ai/DeepSeek-Coder-V2-Lite-Instruct']  # Filter out deepseek model
#Lets filter out pylint and django repos too
df = df[~df['instance_id'].str.contains('pylint|django|pytest', case=False, na=False)]

# Create output directory
output_dir = Path('presentation_diagrams/analysis')
output_dir.mkdir(parents=True, exist_ok=True)

# ========== 1. SUCCESS RATE BY MODEL ==========
def plot_success_rate_by_model():
    # Create model combination label
    df['model_combo'] = df['tests_model'].str.split('/').str[-1].str[:20] + '\n(tests)'

    # Calculate success rate by tests_model
    success_by_model = df.groupby('tests_model').agg({
        'discriminative': 'mean',
        'instance_id': 'count'
    }).reset_index()
    success_by_model.columns = ['model', 'success_rate', 'count']
    success_by_model['success_rate'] *= 100
    success_by_model['model_short'] = success_by_model['model'].str.split('/').str[-1].str[:30]

    fig, ax = plt.subplots(figsize=(10, 6))
    bars = ax.bar(range(len(success_by_model)), success_by_model['success_rate'],
                   color=['#2E86AB', '#A23B72', '#F18F01'])

    # Add count labels on bars
    for i, (bar, count) in enumerate(zip(bars, success_by_model['count'])):
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height + 1,
                f'n={count}', ha='center', va='bottom', fontsize=9)

    ax.set_xlabel('Test Generation Model', fontweight='bold')
    ax.set_ylabel('Success Rate (%)', fontweight='bold')
    ax.set_title('Test Generation Success Rate by Model', fontsize=14, fontweight='bold')
    ax.set_xticks(range(len(success_by_model)))
    ax.set_xticklabels(success_by_model['model_short'], rotation=15, ha='right')
    ax.set_ylim(0, 100)
    ax.axhline(y=50, color='gray', linestyle='--', alpha=0.5, label='50% baseline')
    ax.legend()

    plt.tight_layout()
    plt.savefig(output_dir / '1_success_rate_by_model.png', dpi=300, bbox_inches='tight')
    print(f"✓ Saved: 1_success_rate_by_model.png")
    plt.close()

# ========== 2. CLASSIFICATION DISTRIBUTION BY Model==========
def plot_classification_distribution(): #generate a pie chart and a bar chart showing the distribution of classifications across all instances for each model

    classification_counts = df['classification'].value_counts()

    # Create color map
    colors = {
        'VALID': '#2E7D32',
        'OVERCONSTRAINED': '#D32F2F',
        'NON_DISCRIMINATIVE': '#F57C00',
        'UNRESOLVED': '#7B1FA2',
        'INVERTED': '#C2185B',
        'N/A': '#9E9E9E'
    }

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

    # Pie chart
    wedges, texts, autotexts = ax1.pie(classification_counts,
                                        labels=classification_counts.index,
                                        autopct='%1.1f%%',
                                        colors=[colors.get(x, '#999999') for x in classification_counts.index],
                                        startangle=90)
    for autotext in autotexts:
        autotext.set_color('white')
        autotext.set_fontweight('bold')
    ax1.set_title('Classification Distribution', fontsize=14, fontweight='bold')

    # Bar chart with counts
    bars = ax2.barh(range(len(classification_counts)), classification_counts.values,
                    color=[colors.get(x, '#999999') for x in classification_counts.index])
    ax2.set_yticks(range(len(classification_counts)))
    ax2.set_yticklabels(classification_counts.index)
    ax2.set_xlabel('Count', fontweight='bold')
    ax2.set_title('Classification Counts', fontsize=14, fontweight='bold')

    # Add count labels
    for i, (bar, count) in enumerate(zip(bars, classification_counts.values)):
        ax2.text(count + 0.5, bar.get_y() + bar.get_height()/2,
                str(count), va='center', fontweight='bold')

    plt.tight_layout()
    plt.savefig(output_dir / f'2_classification_distribution.png', dpi=300, bbox_inches='tight')
    print(f"✓ Saved: 2_classification_distribution.png")
    plt.close()

# ========== 3. ATTEMPTS TO SUCCESS ==========
def plot_attempts_analysis():
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

    # Box plot: attempts by status
    status_order = ['success', 'non_discriminative', 'failed', 'no_attempts']
    colors_status = {'success': '#2E7D32', 'non_discriminative': '#F57C00',
                     'failed': '#D32F2F', 'no_attempts': '#9E9E9E'}

    data_for_box = [df[df['status'] == status]['attempts'].values
                    for status in status_order if status in df['status'].values]
    labels_for_box = [s for s in status_order if s in df['status'].values]

    bp = ax1.boxplot(data_for_box, labels=labels_for_box, patch_artist=True)
    for patch, label in zip(bp['boxes'], labels_for_box):
        patch.set_facecolor(colors_status.get(label, '#999999'))

    ax1.set_ylabel('Number of Attempts', fontweight='bold')
    ax1.set_xlabel('Status', fontweight='bold')
    ax1.set_title('Attempts Required by Outcome', fontsize=14, fontweight='bold')
    ax1.set_xticklabels(labels_for_box, rotation=15, ha='right')

    # Histogram: success convergence
    success_df = df[df['status'] == 'success']
    ax2.hist(success_df['attempts'], bins=range(1, 12),
             color='#2E7D32', alpha=0.7, edgecolor='black')
    ax2.set_xlabel('Attempts to Success', fontweight='bold')
    ax2.set_ylabel('Count', fontweight='bold')
    ax2.set_title('Success Convergence Distribution', fontsize=14, fontweight='bold')
    ax2.axvline(success_df['attempts'].median(), color='red',
                linestyle='--', linewidth=2, label=f'Median: {success_df["attempts"].median():.1f}')
    ax2.legend()

    plt.tight_layout()
    plt.savefig(output_dir / '3_attempts_analysis.png', dpi=300, bbox_inches='tight')
    print(f"✓ Saved: 3_attempts_analysis.png")
    plt.close()

# ========== 4. REPOSITORY DIFFICULTY ==========
def plot_repository_difficulty():
    # Extract repository name from instance_id
    df['repo_name'] = df['instance_id'].str.split('__').str[0]

    # Calculate success rate by repo
    repo_stats = df.groupby('repo_name').agg({
        'discriminative': ['mean', 'sum'],
        'instance_id': 'count'
    }).reset_index()
    repo_stats.columns = ['repo', 'success_rate', 'successes', 'total']
    repo_stats['success_rate'] *= 100
    repo_stats = repo_stats.sort_values('success_rate', ascending=True)

    fig, ax = plt.subplots(figsize=(10, 8))

    bars = ax.barh(range(len(repo_stats)), repo_stats['success_rate'])

    # Color bars by difficulty
    for i, bar in enumerate(bars):
        rate = repo_stats.iloc[i]['success_rate']
        if rate >= 50:
            bar.set_color('#2E7D32')  # Green - easy
        elif rate >= 25:
            bar.set_color('#F57C00')  # Orange - medium
        else:
            bar.set_color('#D32F2F')  # Red - hard

    ax.set_yticks(range(len(repo_stats)))
    ax.set_yticklabels(repo_stats['repo'])
    ax.set_xlabel('Success Rate (%)', fontweight='bold')
    ax.set_title('Success Rate by Repository', fontsize=14, fontweight='bold')
    ax.axvline(x=50, color='gray', linestyle='--', alpha=0.5)

    # Add labels with counts
    for i, (bar, row) in enumerate(zip(bars, repo_stats.itertuples())):
        ax.text(row.success_rate + 2, bar.get_y() + bar.get_height()/2,
                f'{row.success_rate:.1f}% ({int(row.successes)}/{int(row.total)})',
                va='center', fontsize=9)

    plt.tight_layout()
    plt.savefig(output_dir / '4_repository_difficulty.png', dpi=300, bbox_inches='tight')
    print(f"✓ Saved: 4_repository_difficulty.png")
    plt.close()

# ========== 5. MODEL PERFORMANCE HEATMAP ==========
def plot_model_heatmap():
    # Create pivot table
    df['tests_model_short'] = df['tests_model'].str.split('/').str[-1].str[:25]
    df['claims_model_short'] = df['claims_model'].str.split('/').str[-1].str[:25]

    # Create discriminative_num if not exists (fixed deprecation warning)
    if 'discriminative_num' not in df.columns:
        df['discriminative_num'] = pd.to_numeric(
            df['discriminative'].replace({True: 1, False: 0, 'True': 1, 'False': 0}).infer_objects(copy=False),
            errors='coerce'
        )

    pivot = df.pivot_table(values='discriminative_num',
                           index='tests_model_short',
                           columns='claims_model_short',
                           aggfunc='mean') * 100

    # Count matrix for annotations
    count_pivot = df.pivot_table(values='discriminative_num',
                                  index='tests_model_short',
                                  columns='claims_model_short',
                                  aggfunc='count')

    fig, ax = plt.subplots(figsize=(10, 8))
    sns.heatmap(pivot, annot=True, fmt='.1f', cmap='RdYlGn',
                vmin=0, vmax=100, cbar_kws={'label': 'Success Rate (%)'},
                linewidths=0.5, ax=ax)

    # Add count annotations
    for i in range(len(pivot.index)):
        for j in range(len(pivot.columns)):
            if not pd.isna(count_pivot.iloc[i, j]):
                ax.text(j + 0.5, i + 0.7, f'n={int(count_pivot.iloc[i, j])}',
                       ha='center', va='center', fontsize=8, color='gray')

    ax.set_title('Success Rate: Tests Model vs Claims Model',
                 fontsize=14, fontweight='bold', pad=20)
    ax.set_xlabel('Claims Extraction Model', fontweight='bold')
    ax.set_ylabel('Test Generation Model', fontweight='bold')

    plt.tight_layout()
    plt.savefig(output_dir / '5_model_performance_heatmap.png', dpi=300, bbox_inches='tight')
    print(f"✓ Saved: 5_model_performance_heatmap.png")
    plt.close()

# ========== 6. FAILURE MODE BREAKDOWN ==========
def plot_failure_modes():
    failed_df = df[df['status'].isin(['failed', 'non_discriminative'])]

    fig, ax = plt.subplots(figsize=(12, 6))

    # Count by classification
    failure_counts = failed_df['classification'].value_counts()

    colors_map = {
        'OVERCONSTRAINED': '#D32F2F',
        'UNRESOLVED': '#7B1FA2',
        'NON_DISCRIMINATIVE': '#F57C00',
        'INVERTED': '#C2185B'
    }

    bars = ax.bar(range(len(failure_counts)), failure_counts.values,
                  color=[colors_map.get(x, '#999999') for x in failure_counts.index])

    ax.set_xticks(range(len(failure_counts)))
    ax.set_xticklabels(failure_counts.index, rotation=15, ha='right')
    ax.set_ylabel('Count', fontweight='bold')
    ax.set_title('Failure Mode Distribution', fontsize=14, fontweight='bold')

    # Add percentage labels
    total = failure_counts.sum()
    for bar, count in zip(bars, failure_counts.values):
        height = bar.get_height()
        pct = (count / total) * 100
        ax.text(bar.get_x() + bar.get_width()/2., height + 0.5,
                f'{count}\n({pct:.1f}%)', ha='center', va='bottom', fontweight='bold')

    plt.tight_layout()
    plt.savefig(output_dir / '6_failure_modes.png', dpi=300, bbox_inches='tight')
    print(f"✓ Saved: 6_failure_modes.png")
    plt.close()

# ========== 7. MULTI-CLAIM ANALYSIS ==========
def plot_multi_claim_analysis():
    # Group by instance and claim
    df['base_instance'] = df['instance_id']
    multi_claim = df[df['claim_id'] != 'C1']

    if len(multi_claim) > 0:
        fig, ax = plt.subplots(figsize=(12, 6))

        # Create discriminative_num if not exists
        if 'discriminative_num' not in df.columns:
            df['discriminative_num'] = pd.to_numeric(
                df['discriminative'].replace({True: 1, False: 0, 'True': 1, 'False': 0}).infer_objects(copy=False),
                errors='coerce'
            )

        # Pivot for heatmap
        pivot = df.pivot_table(values='discriminative_num',
                               index='instance_id',
                               columns='claim_id',
                               aggfunc='mean')

        # Only show instances with multiple claims
        pivot = pivot[pivot.count(axis=1) > 1]

        if len(pivot) > 0:
            sns.heatmap(pivot, annot=True, fmt='.0f', cmap='RdYlGn',
                       vmin=0, vmax=1, cbar_kws={'label': 'Success (1=Yes, 0=No)'},
                       linewidths=0.5, ax=ax)

            ax.set_title('Multi-Claim Success Pattern', fontsize=14, fontweight='bold')
            ax.set_xlabel('Claim ID', fontweight='bold')
            ax.set_ylabel('Instance', fontweight='bold')

            plt.tight_layout()
            plt.savefig(output_dir / '7_multi_claim_analysis.png', dpi=300, bbox_inches='tight')
            print(f"✓ Saved: 7_multi_claim_analysis.png")
            plt.close()
        else:
            print("⊘ Skipped: 7_multi_claim_analysis.png (no multi-claim instances)")
    else:
        print("⊘ Skipped: 7_multi_claim_analysis.png (no multi-claim data)")

# ========== 8. SUMMARY STATISTICS ==========
def plot_summary_stats():
    fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(14, 10))

    # Overall success rate
    overall_success = df['discriminative'].mean() * 100
    ax1.text(0.5, 0.5, f"{overall_success:.1f}%",
             ha='center', va='center', fontsize=60, fontweight='bold',
             color='#2E7D32' if overall_success >= 50 else '#D32F2F')
    ax1.text(0.5, 0.2, "Overall Success Rate",
             ha='center', va='center', fontsize=14, fontweight='bold')
    ax1.text(0.5, 0.1, f"({df['discriminative'].sum():.0f}/{len(df)} instances)",
             ha='center', va='center', fontsize=11)
    ax1.axis('off')

    # Average attempts
    avg_attempts = df['attempts'].mean()
    ax2.text(0.5, 0.5, f"{avg_attempts:.1f}",
             ha='center', va='center', fontsize=60, fontweight='bold', color='#2E86AB')
    ax2.text(0.5, 0.2, "Average Attempts",
             ha='center', va='center', fontsize=14, fontweight='bold')
    ax2.text(0.5, 0.1, f"(median: {df['attempts'].median():.0f})",
             ha='center', va='center', fontsize=11)
    ax2.axis('off')

    # Success attempts vs failure attempts
    success_attempts = df[df['discriminative'] == True]['attempts'].mean()
    failure_attempts = df[df['discriminative'] == False]['attempts'].mean()
    ax3.bar(['Success', 'Failure'], [success_attempts, failure_attempts],
           color=['#2E7D32', '#D32F2F'])
    ax3.set_ylabel('Average Attempts', fontweight='bold')
    ax3.set_title('Attempts: Success vs Failure', fontsize=12, fontweight='bold')
    for i, v in enumerate([success_attempts, failure_attempts]):
        ax3.text(i, v + 0.2, f'{v:.1f}', ha='center', fontweight='bold')

    # Status distribution
    status_counts = df['status'].value_counts()
    ax4.pie(status_counts, labels=status_counts.index, autopct='%1.1f%%',
           colors=['#2E7D32', '#D32F2F', '#F57C00', '#9E9E9E'])
    ax4.set_title('Status Distribution', fontsize=12, fontweight='bold')

    plt.tight_layout()
    plt.savefig(output_dir / '8_summary_statistics.png', dpi=300, bbox_inches='tight')
    print(f"✓ Saved: 8_summary_statistics.png")
    plt.close()

# ========== RUN ALL ==========
if __name__ == "__main__":
    print("Generating analysis plots...")
    print(f"Data loaded: {len(df)} rows\n")

    plot_success_rate_by_model()
    plot_classification_distribution()
    plot_attempts_analysis()
    plot_repository_difficulty()
    plot_model_heatmap()
    plot_failure_modes()
    plot_multi_claim_analysis()
    plot_summary_stats()

    print(f"\n✅ All plots saved to: {output_dir}/")
    print("\nRecommended for PPT (pick 3-4):")
    print("  • 8_summary_statistics.png (overview)")
    print("  • 1_success_rate_by_model.png (model comparison)")
    print("  • 3_attempts_analysis.png (efficiency)")
    print("  • 4_repository_difficulty.png (generalization)")